In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("non-problematic_recipes.parquet", engine="pyarrow")

In [3]:
df.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4
5,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5
6,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6


In [4]:
helper_df = pd.read_csv("tam_eslesme_sonuclari.tsv", sep="\t")

In [5]:
helper_df.head()

,Ingredient,IDs,__index_level_0__,I-NAME tokens,Matched Words,En Uzun Tam Eşleşme
0,low-sodium vegetable,"0, 11434, 14768, 82089, 96363, 105999, 194867,...",0,"▁vegeta, ble",vegetable,vegetable
1,chicken stock,"0, 11, 94, 101, 130, 155, 200, 345, 552, 678, ...",1,"▁chicken, ▁stock","chicken, stock",chicken stock
2,dried brown lentils,"0, 17104, 836751, 838386, 841486, 871267, 8744...",2,"▁brown, ▁lenti, ls","brown, lentils",brown lentils
3,dried french green lentils,"0, 2684, 3173, 3530, 16691",3,"▁fren, ch, ▁green, ▁lenti, ls","french, green, lentils",french green lentils
4,"celery, chopped","0, 1149, 2427, 3530, 5081, 5340, 5891, 8907, 9...",4,"▁cele, ry",celery,celery


In [6]:
mapper_dict = dict(zip(helper_df["Ingredient"], helper_df["En Uzun Tam Eşleşme"]))

In [7]:
mapper_dict["rose's lime juice"]

"rose's lime juice"

In [ ]:
import pandas as pd
import numpy as np
import ast
import json

def extract_ingredient_names(ingredient_data):
    if isinstance(ingredient_data, (np.ndarray, pd.Series)):
        if ingredient_data.size == 0 or pd.isna(ingredient_data).all():
            return []
        ingredient_data = ingredient_data.tolist()

    if ingredient_data is None:
        return []
    if isinstance(ingredient_data, float) and np.isnan(ingredient_data):
        return []

    if isinstance(ingredient_data, list):
        parsed_data = ingredient_data

    elif isinstance(ingredient_data, dict):
        parsed_data = [ingredient_data]

    elif isinstance(ingredient_data, str):
        try:
            parsed_data = ast.literal_eval(ingredient_data)
            if isinstance(parsed_data, dict):
                parsed_data = [parsed_data]
            elif not isinstance(parsed_data, list):
                return []
        except (SyntaxError, ValueError):
            try:
                parsed_data = json.loads(ingredient_data)
                if isinstance(parsed_data, dict):
                    parsed_data = [parsed_data]
                elif not isinstance(parsed_data, list):
                    return []
            except (json.JSONDecodeError, TypeError):
                return []
    else:
        return []

    if not isinstance(parsed_data, list):
        return []

    names = []
    for item in parsed_data:
        if isinstance(item, dict):
            name = item.get('name')
            if name:
                names.append(str(name))

    return names


In [9]:
df["ingredient names"] = df["Ingredients"].apply(extract_ingredient_names)

In [10]:
df.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le..."
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha..."
5,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai..."
6,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,..."


In [11]:
ingredients_map_df = pd.read_csv("ingredients.tsv", sep="\t")

In [12]:
ingredients_map_df.head()

,Ingredients,ID,cleaned_ingredients,splitted_ingredients
0,"['low-sodium vegetable or chicken stock', 'dri...",0,"['low-sodium vegetable or chicken stock', 'dri...","[['low-sodium vegetable', 'chicken stock'], ['..."
1,"['whipping cream', 'onions, chopped', 'salt', ...",1,"['whipping cream', 'onions, chopped', 'salt', ...","[['whipping cream'], ['onions, chopped'], ['sa..."
2,"['fennel bulb (sometimes called anise), stalks...",2,"['fennel bulb , stalks discarded, bulb cut int...","[['fennel bulb , stalks discarded, bulb cut in..."
3,"['extra-virgin olive oil', 'chopped onion', 'd...",3,"['extra-virgin olive oil', 'chopped onion', 'd...","[['extra-virgin olive oil'], ['chopped onion']..."
4,"['12-ounce package frozen spinach soufflé, th...",4,"['package frozen spinach souffle, thawed', 'ex...","[['package frozen spinach souffle, thawed'], [..."


In [13]:
ingredients_map_df.drop(columns=["Ingredients", "cleaned_ingredients"], inplace=True)

In [14]:
ingredients_map_df.head()

,ID,splitted_ingredients
0,0,"[['low-sodium vegetable', 'chicken stock'], ['..."
1,1,"[['whipping cream'], ['onions, chopped'], ['sa..."
2,2,"[['fennel bulb , stalks discarded, bulb cut in..."
3,3,"[['extra-virgin olive oil'], ['chopped onion']..."
4,4,"[['package frozen spinach souffle, thawed'], [..."


In [15]:
df_result = df.merge(ingredients_map_df, on='ID', how='left')

In [16]:
df_result.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names,splitted_ingredients
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ...","[['low-sodium vegetable', 'chicken stock'], ['..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le...","[['whipping cream'], ['onions, chopped'], ['sa..."
2,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha...","[['package frozen spinach souffle, thawed'], [..."
3,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai...","[['fresh basil leaves'], ['mayonaisse'], ['but..."
4,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,...","[['red-skinned potatoes, each cut into wedges'..."


In [17]:
df_result.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names,splitted_ingredients
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ...","[['low-sodium vegetable', 'chicken stock'], ['..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le...","[['whipping cream'], ['onions, chopped'], ['sa..."
2,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha...","[['package frozen spinach souffle, thawed'], [..."
3,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai...","[['fresh basil leaves'], ['mayonaisse'], ['but..."
4,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,...","[['red-skinned potatoes, each cut into wedges'..."


In [18]:
def map_2d_list(two_d_list, mapper):
    # Example: two_d_list = [
    #   ['low sodium vegetable', 'chicken stock'],
    #   ['dried brown lentils'],
    #   ['dried French green lentils']
    # ]
    mapped_list = []
    for sublist in two_d_list:       # sublist example: ['low sodium vegetable', 'chicken stock']
        new_sublist = []
        for item in sublist:        # item example: 'low sodium vegetable'
            new_sublist.append(mapper_dict.get(item, item))
        mapped_list.append(new_sublist)
    return mapped_list

In [19]:
df_result['parsed_ingredients'] = df_result['splitted_ingredients'].apply(lambda x: ast.literal_eval(x))

In [20]:
df_result.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names,splitted_ingredients,parsed_ingredients
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ...","[['low-sodium vegetable', 'chicken stock'], ['...","[[low-sodium vegetable, chicken stock], [dried..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le...","[['whipping cream'], ['onions, chopped'], ['sa...","[[whipping cream], [onions, chopped], [salt], ..."
2,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha...","[['package frozen spinach souffle, thawed'], [...","[[package frozen spinach souffle, thawed], [ex..."
3,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai...","[['fresh basil leaves'], ['mayonaisse'], ['but...","[[fresh basil leaves], [mayonaisse], [butter, ..."
4,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,...","[['red-skinned potatoes, each cut into wedges'...","[[red-skinned potatoes, each cut into wedges],..."


In [21]:
df_temp = df_result.head()

In [22]:
df_temp['mapped_ingredients'] = df_temp['parsed_ingredients'].apply(lambda x: map_2d_list(x, mapper_dict))

C:\Users\abdur\AppData\Local\Temp\ipykernel_26824\3810931128.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['mapped_ingredients'] = df_temp['parsed_ingredients'].apply(lambda x: map_2d_list(x, mapper_dict))


In [23]:
df_temp.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names,splitted_ingredients,parsed_ingredients,mapped_ingredients
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ...","[['low-sodium vegetable', 'chicken stock'], ['...","[[low-sodium vegetable, chicken stock], [dried...","[[vegetable, chicken stock], [brown lentils], ..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le...","[['whipping cream'], ['onions, chopped'], ['sa...","[[whipping cream], [onions, chopped], [salt], ...","[[whipping cream], [onions], [salt], [bay leav..."
2,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha...","[['package frozen spinach souffle, thawed'], [...","[[package frozen spinach souffle, thawed], [ex...","[[spinach souffle], [extra-wide egg noodles], ..."
3,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai...","[['fresh basil leaves'], ['mayonaisse'], ['but...","[[fresh basil leaves], [mayonaisse], [butter, ...","[[basil leaves], [mayonaisse], [butter], [baco..."
4,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,...","[['red-skinned potatoes, each cut into wedges'...","[[red-skinned potatoes, each cut into wedges],...","[[red-skinned potatoes], [baby carrots], [aspa..."


In [24]:
df_result['mapped_ingredients'] = df_result['parsed_ingredients'].apply(lambda x: map_2d_list(x, mapper_dict))

In [25]:
df_result.tail()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names,splitted_ingredients,parsed_ingredients,mapped_ingredients
1072366,Maple Apple Baked Beans,1. Place beans in soup kettle; add water to co...,"[{""name"":""dried navy beans"",""quantity"":91,""uni...",375.0,518.1,19.0,17.3,garlish,72.3,1174475,"[dried navy beans, water, bacon, onion, salt, ...","[['dried navy beans'], ['water'], ['bacon'], [...","[[dried navy beans], [water], [bacon], [onion]...","[[navy beans], [water], [bacon], [onion], [sal..."
1072367,Blackberry Orange Scones,1. Sift about 2 cups of flour onto a piece of ...,"[{""name"":""unbleached flour"",""quantity"":91,""uni...",27.0,244.8,9.1,5.1,dessert,35.4,1174476,"[unbleached flour, baking soda, butter, orange...","[['unbleached flour'], ['baking soda'], ['butt...","[[unbleached flour], [baking soda], [butter], ...","[[flour], [baking soda], [butter], [orange zes..."
1072368,Slow Cooker Garlic Chicken With Rosemary,"1. Place rosemary springs, 1 lemon half, celer...","[{""name"":""roasting chickens"",""quantity"":91,""un...",440.0,566.2,38.9,43.2,main dish,9.3,1174477,"[roasting chickens, lemons, rosemary sprigs, p...","[['roasting chickens'], ['lemons'], ['rosemary...","[[roasting chickens], [lemons], [rosemary spri...","[[roasting chickens], [lemons], [rosemary], [p..."
1072369,Kapusta ( Cabbage and Kielbasa ),1. Saute bacon in large pan until browned. Le...,"[{""name"":""cabbage"",""quantity"":91,""unit"":""cups""...",NaN,688.0,48.5,25.6,main dish,39.2,1174478,"[cabbage, cabbage, kielbasa, onions, water, ba...","[['cabbage'], ['cabbage'], ['kielbasa'], ['oni...","[[cabbage], [cabbage], [kielbasa], [onions], [...","[[cabbage], [cabbage], [kielbasa], [onions], [..."
1072370,Yellow or Zucchini Squash Pie,"1. Melt butter in skillet. Add squash, onions,...","[{""name"":""zucchini"",""quantity"":91,""unit"":""cups...",40.0,411.0,28.1,15.4,garlish,25.1,1174479,"[zucchini, onion, butter, oregano, basil, eggs...","[['zucchini'], ['onion'], ['butter'], ['oregan...","[[zucchini], [onion], [butter], [oregano], [ba...","[[zucchini], [onion], [butter], [oregano], [ba..."


In [26]:
df_result["concetenated"] = ""

In [27]:
df_result.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names,splitted_ingredients,parsed_ingredients,mapped_ingredients,concetenated
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ...","[['low-sodium vegetable', 'chicken stock'], ['...","[[low-sodium vegetable, chicken stock], [dried...","[[vegetable, chicken stock], [brown lentils], ...",
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le...","[['whipping cream'], ['onions, chopped'], ['sa...","[[whipping cream], [onions, chopped], [salt], ...","[[whipping cream], [onions], [salt], [bay leav...",
2,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha...","[['package frozen spinach souffle, thawed'], [...","[[package frozen spinach souffle, thawed], [ex...","[[spinach souffle], [extra-wide egg noodles], ...",
3,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai...","[['fresh basil leaves'], ['mayonaisse'], ['but...","[[fresh basil leaves], [mayonaisse], [butter, ...","[[basil leaves], [mayonaisse], [butter], [baco...",
4,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,...","[['red-skinned potatoes, each cut into wedges'...","[[red-skinned potatoes, each cut into wedges],...","[[red-skinned potatoes], [baby carrots], [aspa...",


In [28]:
import math

invalid_values = {"", "none", "nan", "null", "or", "to"}

def flatten_and_clean(two_d_list):
    flattened_list = []
    
    if not two_d_list:
        return flattened_list
    
    for sub in two_d_list:
        if not isinstance(sub, list):
            sub = [sub]
        
        cleaned = []
        for elem in sub:
            if elem is None or (isinstance(elem, float) and math.isnan(elem)):
                continue
                
            elem_str = str(elem).strip()
            if elem_str.lower() in invalid_values:
                continue
                
            cleaned.append(elem_str)
        
        if not cleaned:
            flattened_list.append("delete")
        elif len(cleaned) == 1:
            flattened_list.append(cleaned[0])
        else:
            flattened_list.append(" or ".join(cleaned))
    
    for idx, item in enumerate(flattened_list):
        if item is None or (isinstance(item, float) and math.isnan(item)):
            flattened_list[idx] = "delete"
        elif isinstance(item, str) and item.strip().lower() in {"", "none", "nan", "null"}:
            flattened_list[idx] = "delete"
    
    return flattened_list


In [29]:
df_result["concetenated"] = df_result["mapped_ingredients"].apply(flatten_and_clean)

In [30]:
print(df_result["ingredient names"].iloc[717619])

['olive oil', 'chicken', 'whole grain mustard', 'carrot', 'onion', 'unsalted butter', 'carrot', 'onion', 'leek', 'chicken stock', 'dry white wine', 'white wine vinegar', 'parsley', 'tarragon', 'chive', 'fresh ground black pepper']


In [31]:
print(df_result["ID"].iloc[572664])

652908


In [32]:
print(df_result["splitted_ingredients"].iloc[717619])

[['olive oil'], ['chicken'], ['whole grain mustard'], ['carrot'], ['onion'], ['unsalted butter'], ['carrot'], ['onion'], ['leek'], ['chicken stock'], ['dry white wine'], ['white wine vinegar'], ['parsley'], ['tarragon'], ['chive'], ['fresh ground black pepper']]


In [33]:
print(df_result["mapped_ingredients"].iloc[717619])

[['olive oil'], ['chicken'], ['mustard'], ['carrot'], ['onion'], ['unsalted butter'], ['carrot'], ['onion'], ['leek'], ['chicken stock'], ['white wine'], ['white wine vinegar'], ['parsley'], ['tarragon'], ['chive'], ['black pepper']]


In [34]:
print(df_result["concetenated"].iloc[160661])  

['spice cake mix', 'water', 'vegetable oil', 'egg', 'mincemeat', 'premium eggnog', 'whipped dessert topping mix']


In [35]:
mask = df_result.apply(lambda row: len(row["ingredient names"]) != len(row["concetenated"]), axis=1)

In [36]:
print(df_result[mask].index.to_list())

[]


In [37]:
df_result.columns

Index(['Name', 'Instructions', 'Ingredients', 'Total Time', 'Calories', 'Fat',
       'Protein', 'Label', 'Carbohydrate', 'ID', 'ingredient names',
       'splitted_ingredients', 'parsed_ingredients', 'mapped_ingredients',
       'concetenated'],
      dtype='object')

In [38]:
import pandas as pd
import numpy as np
import ast
import json
import math

def is_null_value(val):
    if val is None:
        return True
    if isinstance(val, float) and math.isnan(val):
        return True
    if isinstance(val, str):
        stripped = val.strip()
        if stripped == "":
            return True
        if stripped.lower() in {"null", "none", "nan"}:
            return True
    return False

def update_ingredient_data(ingredient_data, concetenated_values):
    if isinstance(ingredient_data, (np.ndarray, pd.Series)):
        if ingredient_data.size == 0 or pd.isna(ingredient_data).all():
            return ([], 0)
        ingredient_data = ingredient_data.tolist()
    
    if ingredient_data is None:
        return ([], 0)
    if isinstance(ingredient_data, float) and np.isnan(ingredient_data):
        return ([], 0)
    
    if isinstance(ingredient_data, list):
        parsed_data = ingredient_data
    elif isinstance(ingredient_data, dict):
        parsed_data = [ingredient_data]
    elif isinstance(ingredient_data, str):
        try:
            parsed_data = ast.literal_eval(ingredient_data)
            if isinstance(parsed_data, dict):
                parsed_data = [parsed_data]
            elif not isinstance(parsed_data, list):
                return ([], 0)
        except (SyntaxError, ValueError):
            try:
                parsed_data = json.loads(ingredient_data)
                if isinstance(parsed_data, dict):
                    parsed_data = [parsed_data]
                elif not isinstance(parsed_data, list):
                    return ([], 0)
            except (json.JSONDecodeError, TypeError):
                return ([], 0)
    else:
        return ([], 0)
    
    if not isinstance(parsed_data, list):
        return ([], 0)
        
    updated_data = []
    null_count = 0  
    
    c_idx = 0
    
    for item_dict in parsed_data:
        if not isinstance(item_dict, dict):
            continue
        
        item = dict(item_dict)
        
        if "name" in item and is_null_value(item["name"]):
            null_count += 1
            continue
        
        if c_idx < len(concetenated_values):
            new_val = concetenated_values[c_idx]
            c_idx += 1  
            if isinstance(new_val, str):
                cleaned_new_val = new_val.lower().strip()
                if cleaned_new_val == "delete":
                    continue
                if cleaned_new_val in {"null", "none", "nan", ""}:
                    null_count += 1
                    continue
                item["name"] = new_val
            else:
                item["name"] = new_val
        if "name" in item and is_null_value(item["name"]):
            null_count += 1
            continue
        
        updated_data.append(item)
    
    return (updated_data, null_count)

def apply_update(row):
    updated_data, null_count = update_ingredient_data(row["Ingredients"], row["concetenated"])
    row["Corrected Ingredients"] = updated_data
    row["Null Count"] = null_count
    return row

In [39]:
df_result["Null Count"] = 0

In [40]:
df_result = df_result.apply(apply_update, axis=1)

In [41]:
df_result["Corrected Ingredient Names"] = df_result["Corrected Ingredients"].apply(extract_ingredient_names)

In [42]:
import ast
import json

def find_mismatched_indices(df_result):
    mismatched_indices = []
    
    for idx, row in df_result.iterrows():
        ingredients_val = row["Ingredients"]
        concetenated = row["concetenated"]
        corrected_names = row["Corrected Ingredient Names"]
        null_count = row["Null Count"]

        if isinstance(ingredients_val, str):
            try:
                ingredient_list = ast.literal_eval(ingredients_val)
                if not isinstance(ingredient_list, list):
                    ingredient_list = []
            except Exception:
                try:
                    ingredient_list = json.loads(ingredients_val)
                    if not isinstance(ingredient_list, list):
                        ingredient_list = []
                except Exception:
                    ingredient_list = []
        elif isinstance(ingredients_val, list):
            ingredient_list = ingredients_val
        else:
            ingredient_list = []
        
        if not isinstance(concetenated, list):
            concetenated = [concetenated]
        if not isinstance(corrected_names, list):
            corrected_names = [corrected_names]

        total_ingredients = len(ingredient_list)
        delete_count = concetenated.count("delete")
        corrected_count = len(corrected_names)

        if total_ingredients != (delete_count + corrected_count + null_count):
            mismatched_indices.append(idx)
    
    return mismatched_indices


In [43]:
# Integrity control
mismatches = find_mismatched_indices(df_result)
print("Problematic indexes:", len(mismatches))

Problematic indexes: 0


In [44]:
empty_indices = df_result[df_result["ingredient names"].apply(lambda x: isinstance(x, list) and len(x) == 0)].index.tolist()
df_result_final = df_result.drop(empty_indices)

In [45]:
len(empty_indices)

1362

In [46]:
len(df_result)

1072371

In [47]:
len(df_result_final)

1071009

In [48]:
print(mismatches)

[]


In [49]:
count_nan = df_result["concetenated"].apply(
    lambda lst: sum(
        math.isnan(x) for x in lst 
        if isinstance(x, float)
    )
).sum()

print("Toplam float NaN sayısı:", count_nan)

Toplam float NaN sayısı: 0


In [50]:
df_final = pd.DataFrame(columns=["ID", "Name", "Instructions", "Ingredients", "Total Time", "Calories", "Fat", "Protein", "Carbohydrate", "Label"])

In [51]:
df_final.head()

,ID,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Carbohydrate,Label


In [52]:
df_final["ID"] = df_result_final["ID"]
df_final["Name"] = df_result_final["Name"]
df_final["Instructions"] = df_result_final["Instructions"]
df_final["Ingredients"] = df_result_final["Corrected Ingredients"]
df_final["Total Time"] = df_result_final["Total Time"]
df_final["Calories"] = df_result_final["Calories"]
df_final["Fat"] = df_result_final["Fat"]
df_final["Protein"] = df_result_final["Protein"]
df_final["Carbohydrate"] = df_result_final["Carbohydrate"]
df_final["Category"] = df_result_final["Label"]

In [53]:
df_final.head()

,ID,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Carbohydrate,Label,Category
0,0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{'name': 'vegetable or chicken stock', 'quant...",30.0,426.0,7.0,30.0,NaN,NaN,garlish
1,1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{'name': 'whipping cream', 'quantity': 5.5, '...",177.0,403.0,23.0,18.0,NaN,NaN,main dish
2,4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{'name': 'spinach souffle', 'quantity': 1.0, ...",55.0,547.0,32.0,20.0,NaN,NaN,main dish
3,5,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{'name': 'basil leaves', 'quantity': 10.5, 'u...",8.0,948.0,79.0,19.0,NaN,NaN,main dish
4,6,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{'name': 'red-skinned potatoes', 'quantity': ...",10.0,NaN,NaN,NaN,NaN,NaN,main dish


In [55]:
df_result_final.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names,splitted_ingredients,parsed_ingredients,mapped_ingredients,concetenated,Null Count,Corrected Ingredients,Corrected Ingredient Names
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ...","[['low-sodium vegetable', 'chicken stock'], ['...","[[low-sodium vegetable, chicken stock], [dried...","[[vegetable, chicken stock], [brown lentils], ...","[vegetable or chicken stock, brown lentils, fr...",0,"[{'name': 'vegetable or chicken stock', 'quant...","[vegetable or chicken stock, brown lentils, fr..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le...","[['whipping cream'], ['onions, chopped'], ['sa...","[[whipping cream], [onions, chopped], [salt], ...","[[whipping cream], [onions], [salt], [bay leav...","[whipping cream, onions, salt, bay leaves, clo...",0,"[{'name': 'whipping cream', 'quantity': 5.5, '...","[whipping cream, onions, salt, bay leaves, clo..."
2,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha...","[['package frozen spinach souffle, thawed'], [...","[[package frozen spinach souffle, thawed], [ex...","[[spinach souffle], [extra-wide egg noodles], ...","[spinach souffle, extra-wide egg noodles, crea...",0,"[{'name': 'spinach souffle', 'quantity': 1.0, ...","[spinach souffle, extra-wide egg noodles, crea..."
3,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai...","[['fresh basil leaves'], ['mayonaisse'], ['but...","[[fresh basil leaves], [mayonaisse], [butter, ...","[[basil leaves], [mayonaisse], [butter], [baco...","[basil leaves, mayonaisse, butter, bacon strip...",0,"[{'name': 'basil leaves', 'quantity': 10.5, 'u...","[basil leaves, mayonaisse, butter, bacon strip..."
4,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,...","[['red-skinned potatoes, each cut into wedges'...","[[red-skinned potatoes, each cut into wedges],...","[[red-skinned potatoes], [baby carrots], [aspa...","[red-skinned potatoes, baby carrots, asparagus...",0,"[{'name': 'red-skinned potatoes', 'quantity': ...","[red-skinned potatoes, baby carrots, asparagus..."


In [56]:
df.head()

,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Label,Carbohydrate,ID,ingredient names
0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{""name"": ""low-sodium vegetable or chicken sto...",30.0,426.0,7.0,30.0,garlish,NaN,0,"[low-sodium vegetable or chicken stock, dried ..."
1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{""name"": ""whipping cream"", ""quantity"": 5.5, ""...",177.0,403.0,23.0,18.0,main dish,NaN,1,"[whipping cream, onions, chopped, salt, bay le..."
4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{""name"": ""12-ounce package frozen spinach sou...",55.0,547.0,32.0,20.0,main dish,NaN,4,"[12-ounce package frozen spinach soufflé, tha..."
5,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{""name"": ""(lightly packed) fresh basil leaves...",8.0,948.0,79.0,19.0,main dish,NaN,5,"[(lightly packed) fresh basil leaves, mayonnai..."
6,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{""name"": ""red-skinned potatoes, each cut into...",10.0,NaN,NaN,NaN,main dish,NaN,6,"[red-skinned potatoes, each cut into 8 wedges,..."


In [57]:
df_final['Ingredients'] = df_final['Ingredients'].astype(str)
df_final.to_parquet("changed_and_non_problematic_ingredients.parquet", engine="pyarrow")

In [63]:
print(len(df_final))

1071009
